# 03 · Train YOLO targets

**Purpose.** Export the member images as Ultralytics classification folders and train YOLO26n-cls from scratch: C6 (no augmentation, checkpoints at 5, 10, 20, 40, 60, 80, 100 epochs) and C7 (augmentation, dropout, early stopping).

**Inputs.** Split file, CIFAR pool, a CUDA GPU if available (CPU fallback).

**Outputs.** `data/yolo_ds/<model_id>/`, `models/yolo/<model_id>/weights/`, `models` rows for C6 and C7.

**Syllabus topics.** Lightweight models (YOLO Nano); energy-efficient edge models (parameters, size, ms/image).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU fallback)")

In [ ]:
from mia.training import Workspace, run_jobs

ws = Workspace.open(need_features=False)
exp = ws.exp
jobs = [(cid, seed, None) for cid in ("C6", "C7") for seed in exp.seeds_yolo]
summary = run_jobs(ws, jobs)

In [ ]:
from mia import db

m = db.models_df(ws.conn, role="target")
table = m[m["config_id"].isin(["C6", "C7"])][["model_id", "config_id", "seed", "epoch", "train_acc", "test_acc", "gap_acc",
                                              "loss_ratio", "n_params", "model_bytes", "infer_ms_per_img", "train_seconds"]]
table.round(4)

In [ ]:
for _, r in table.iterrows():
    print(f"{r['model_id']:<22} GPU/CPU time to this checkpoint: {r['train_seconds'] / 60:6.1f} min")
print(f"Trained {summary['trained']}, skipped {summary['skipped']}, failed {summary['failed']}")

## Results
The printed times give the training cost per model; the table shows C6's gap growing with epochs and C7's regularised recipe keeping it small.